# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [1]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


In [2]:
df_clean =spotify_raw.copy()

### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [3]:
df_clean['Release Date'] = pd.to_datetime(df_clean['Release Date'], errors='coerce')
df_clean['Año'] = df_clean['Release Date'].dt.year
df_clean['Mes'] = df_clean['Release Date'].dt.month
df_clean['Dia'] = df_clean['Release Date'].dt.day

In [4]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = df_clean[['Release Date', 'Año', 'Mes', 'Dia']].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = dim_tiempo.index + 1

display(dim_tiempo.head(3))

,Release Date,Año,Mes,Dia,id_tiempo
0,2024-04-26,2024,4,26,1
1,2024-05-04,2024,5,4,2
2,2024-03-19,2024,3,19,3


In [5]:
# 2. Crear dim_pista
dim_pista = dim_pista = df_clean[['Track', 'Artist', 'Album Name', 'ISRC', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_pista['id_pista'] = dim_pista.index + 1
display(dim_pista.head(3))

,Track,Artist,Album Name,ISRC,Explicit Track,id_pista
0,MILLION DOLLAR BABY,Tommy Richman,Million Dollar Baby - Single,QM24S2402528,0,1
1,Not Like Us,Kendrick Lamar,Not Like Us,USUG12400910,1,2
2,i like the way you kiss me,Artemas,I like the way you kiss me,QZJ842400387,0,3


In [6]:
columnas_metricas = [
    'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach',
    'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts',
    'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count',
    'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach',
    'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations',
    'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track'
]

for col in columnas_metricas:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.replace(',', '')
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0)

In [9]:
# 3. Crear fact_streams
fact_streams = fact_streams = df_clean.merge(dim_tiempo, on=['Release Date', 'Año', 'Mes', 'Dia'], how='left')
fact_streams = fact_streams.merge(dim_pista, on=['Track', 'Artist', 'Album Name', 'ISRC', 'Explicit Track'], how='left')
display(fact_streams.head(3))

,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track,Año,Mes,Dia,id_tiempo,id_pista
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,2024-04-26,QM24S2402528,1,725.4,390470936.0,30716.0,196631588.0,...,22931.0,4818457.0,2669262.0,0.0,0,2024,4,26,1,1
1,Not Like Us,Not Like Us,Kendrick Lamar,2024-05-04,USUG12400910,2,545.9,323703884.0,28113.0,174597137.0,...,28444.0,6623075.0,1118279.0,0.0,1,2024,5,4,2,2
2,i like the way you kiss me,I like the way you kiss me,Artemas,2024-03-19,QZJ842400387,3,538.4,601309283.0,54331.0,211607669.0,...,5639.0,7208651.0,5285340.0,0.0,0,2024,3,19,3,3


In [10]:
columnas_fact = ['id_pista', 'id_tiempo'] + [c for c in columnas_metricas if c in fact_streams.columns]
fact_streams = fact_streams[columnas_fact]

In [11]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = cubo_olap = fact_streams.merge(dim_pista, on='id_pista', how='left') \
                        .merge(dim_tiempo, on='id_tiempo', how='left')

display(cubo_olap.head(3))

,id_pista,id_tiempo,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,TikTok Posts,...,Explicit Track_x,Track,Artist,Album Name,ISRC,Explicit Track_y,Release Date,Año,Mes,Dia
0,1,1,725.4,390470936.0,30716.0,196631588.0,92.0,84274754.0,1713126.0,5767700.0,...,0,MILLION DOLLAR BABY,Tommy Richman,Million Dollar Baby - Single,QM24S2402528,0,2024-04-26,2024,4,26
1,2,2,545.9,323703884.0,28113.0,174597137.0,92.0,116347040.0,3486739.0,674700.0,...,1,Not Like Us,Kendrick Lamar,Not Like Us,USUG12400910,1,2024-05-04,2024,5,4
2,3,3,538.4,601309283.0,54331.0,211607669.0,92.0,122599116.0,2228730.0,3025400.0,...,0,i like the way you kiss me,Artemas,I like the way you kiss me,QZJ842400387,0,2024-03-19,2024,3,19


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

Se crearon 2 tablas de dimensiones principales (dim_pista y dim_tiempo) debido al diseño del dataset, donde el detalle de artista y álbum venía desnormalizado junto a la pista. Se podria agregar la dimension (dim_plataforma) y ver como responde.

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [12]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: roll up
respuesta_1 = cubo_olap.groupby('Artist')['Spotify Streams'].sum().reset_index()

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: slice
respuesta_2 = cubo_olap[cubo_olap['Año'] == 2024]

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: drill down
respuesta_3 = cubo_olap.groupby(['Año', 'Mes'])['Spotify Streams'].sum().reset_index()

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: dice
respuesta_4 = cubo_olap[(cubo_olap['Explicit Track'] == 1) & (cubo_olap['Año'] == 2024)]
# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: roll up
respuesta_5 = cubo_olap.groupby(['Año'])['Track Score'].mean().reset_index()

KeyError: 'Explicit Track'

In [13]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: slice
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: drill down
respuesta_7 = cubo_olap.groupby(['Artist', 'Track'])['YouTube Views'].sum().reset_index()

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: dice
respuesta_8 = cubo_olap[(cubo_olap['Mes'] == 12) & (cubo_olap['Año'] == 2023)]

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: roll up
respuesta_9 = cubo_olap.groupby('Mes')['Spotify Streams'].sum().reset_index()

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: dice
respuesta_10 = cubo_olap[(cubo_olap['Artist'] == 'Taylor Swift') & (cubo_olap['Explicit Track'] == 0) & (cubo_olap['Año'] == 2024)]

KeyError: 'Explicit Track'